In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub

import requests
from io import StringIO
import re

c:\Users\esteb\miniconda3\envs\Programar\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Diccionario de datos - tablas

### goals_match (Goles anotados en partidos internacionales)
Datos históricos de goles anotados en partidos de fútbol internacional desde 1916 hasta marzo 2026.
* **date**: Fecha del partido (formato YYYY-MM-DD)
* **home_team**: Nombre del equipo local
* **away_team**: Nombre del equipo visitante
* **team**: Equipo que anotó el gol (local o visitante)
* **scorer**: Nombre del jugador que anotó el gol
* **minute**: Minuto en el que se anotó el gol (float, puede incluir minutos de prórroga)
* **own_goal**: Booleano indicando si fue gol en contra (True/False)
* **penalty**: Booleano indicando si fue de penalti (True/False)
* **Total**: 47.601 registros de goles

### elo_match (Ratings ELO por equipo y fecha)
Ratings ELO históricos de selecciones nacionales calculados desde 1872. Estos ratings se utilizan como variable de entrada para el modelo, calculando la diferencia de ELO entre equipos en el momento del partido.
* **date**: Fecha de la actualización del rating (YYYY-MM-DD)
* **team**: Nombre del país/selección (nombres estandarizados en inglés)
* **rating**: Rating ELO calculado en ese punto temporal (float)
* **change**: Puntos ganados o perdidos con respecto al rating anterior (int)
* **Total**: 6.678 registros de actualizaciones

### fc2026 (Base de datos de jugadores FIFA 26)
Datos de 18.405 jugadores profesionales del videojuego FIFA 26 (actualización 2025-09-19). Incluye información demográfica, datos de club, características técnicas y posiciones en el campo.
* **Información del jugador**: player_id, short_name, long_name, dob, nationality_name, preferred_foot
* **Datos físicos**: age, height_cm, weight_kg, body_type, real_face
* **Información de club**: club_name, club_position, club_jersey_number, club_joined_date, club_contract_valid_until_year
* **Información internacional**: nation_position, nation_jersey_number, international_reputation
* **Valoración**: overall, potential, value_eur, wage_eur, release_clause_eur
* **Atributos técnicos** (6 principales): pace, shooting, passing, dribbling, defending, physic
* **Atributos detallados** (60+ subcategorías): movimiento, defensa, ataque, mentalidad, golpeo, portería, cruces, finalizaciones, pases, regates, posicionamiento, etc.
* **Posiciones**: ratings individuales por cada posición en el campo (LS, ST, RS, LW, CAM, CM, CDM, LB, CB, RB, GK, etc.)
* **Total**: 18.405 registros de jugadores

### groups_fifa (Fixture de fase de grupos - Copa Mundial 2026)
Información de los 12 grupos de 4 equipos y sus 72 partidos de fase de grupos definidos en el sorteo del 5 de diciembre de 2025 en Washington D.C.
* **group**: Identificador del grupo (A-L)
* **match_number**: Número secuencial del partido en la fase de grupos (1-72)
* **team_1**: Primer equipo del partido (equipo local)
* **team_2**: Segundo equipo del partido (equipo visitante)
* **Total**: 72 registros (fixture de fase de grupos)
* **Nota**: Incluye información de los 12 grupos de 4 equipos (6 partidos por grupo)

In [20]:
url_results = "https://raw.githubusercontent.com/martj42/international_results/master/goalscorers.csv"
url_goalscorers = "https://raw.githubusercontent.com/martj42/international_results/master/goalscorers.csv"

C:\Users\esteb\AppData\Local\Temp\ipykernel_7808\2641676526.py:8: DtypeWarning: Columns (0: player_tags) have mixed types. Specify dtype option on import or set low_memory=False.
  fc2026= pd.read_csv("FC26_20250921.csv")


### Data teams FIFA 2026

In [32]:
url = "https://en.wikipedia.org/wiki/2026_FIFA_World_Cup"

headers = {
    "User-Agent": "Mozilla/5.0"
}

html = requests.get(url, headers=headers).text

tablas = pd.read_html(StringIO(html))

partidos = []
grupo_actual = None

for i, tabla in enumerate(tablas):
    
    # Detectar tabla de grupo
    if "Teamvte" in tabla.columns:
        grupo_actual = chr(65 + len(set([p["group"] for p in partidos])))
    
    # Detectar tabla de partido: 1 fila, 3 columnas, columna central tipo Match X
    if tabla.shape == (1, 3):
        cols = list(tabla.columns)
        
        if any("Match" in str(c) for c in cols):
            team_1 = str(cols[0])
            match_col = str(cols[1])
            team_2 = str(cols[2])
            
            match_number = re.search(r"Match\s*(\d+)", match_col)
            match_number = int(match_number.group(1)) if match_number else None
            
            partidos.append({
                "group": grupo_actual,
                "match_number": match_number,
                "team_1": team_1,
                "team_2": team_2
            })



#### Data loaded

In [33]:
results_match = pd.read_csv(url_results)
goals_match = pd.read_csv(url_goalscorers)
elo_match= pd.read_csv("eloratings.csv")
fc2026= pd.read_csv("FC26_20250921.csv")
groups_fifa = pd.DataFrame(partidos)


C:\Users\esteb\AppData\Local\Temp\ipykernel_7808\913261139.py:4: DtypeWarning: Columns (0: player_tags) have mixed types. Specify dtype option on import or set low_memory=False.
  fc2026= pd.read_csv("FC26_20250921.csv")


In [31]:
results_match.head()

,date,home_team,away_team,team,scorer,minute,own_goal,penalty
0,1916-07-02,Chile,Uruguay,Uruguay,José Piendibene,44.0,False,False
1,1916-07-02,Chile,Uruguay,Uruguay,Isabelino Gradín,55.0,False,False
2,1916-07-02,Chile,Uruguay,Uruguay,Isabelino Gradín,70.0,False,False
3,1916-07-02,Chile,Uruguay,Uruguay,José Piendibene,75.0,False,False
4,1916-07-06,Argentina,Chile,Argentina,Alberto Ohaco,2.0,False,False
